In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ==============================
# House Prices Regression Script
# ==============================

# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Metrics
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Stats
from scipy.stats import skew


# ==============================
# Load Data
# ==============================
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test  = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain info:")
train.info()

print("\nTest info:")
test.info()


# ==============================
# Redundant / Exploratory Checks
# ==============================
print("\nMissing values (train):")
print(train.isnull().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows (train):", train.duplicated().sum())

print("\nTrain head:")
print(train.head())

print("\nTrain tail:")
print(train.tail())

print("\nColumns:")
print(train.columns.tolist())


# ==============================
# Exploratory Data Analysis
# ==============================
sns.histplot(train["SalePrice"], kde=True)
plt.title("SalePrice Distribution")
plt.show()

sns.histplot(np.log(train["SalePrice"]), kde=True)
plt.title("Log(SalePrice) Distribution")
plt.show()

plt.figure(figsize=(6, 4))
sns.scatterplot(x=train["GrLivArea"], y=train["SalePrice"])
plt.title("GrLivArea vs SalePrice")
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(x=train["OverallQual"], y=train["SalePrice"])
plt.title("OverallQual vs SalePrice")
plt.show()

plt.figure(figsize=(6, 4))
sns.scatterplot(x=train["YearBuilt"], y=train["SalePrice"])
plt.title("YearBuilt vs SalePrice")
plt.show()


# ==============================
# Target & Data Merge
# ==============================
y = np.log(train["SalePrice"])
train.drop("SalePrice", axis=1, inplace=True)

all_data = pd.concat([train, test], axis=0).reset_index(drop=True)

print("\nCombined data shape:", all_data.shape)


# ==============================
# Missing Value Handling
# ==============================
print("\nTop missing columns before handling:")
print(all_data.isnull().sum().sort_values(ascending=False).head(20))

none_cols = [
    "Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"
]

for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

# Neighborhood-wise median imputation
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"] \
                                   .transform(lambda x: x.fillna(x.median()))

# Remaining missing values
all_data.fillna(0, inplace=True)

print("\nRemaining missing values after handling:")
print(all_data.isnull().sum().sum())


# ==============================
# Feature Engineering
# ==============================
all_data["HouseAge"] = all_data["YrSold"] - all_data["YearBuilt"]
all_data["RemodAge"] = all_data["YrSold"] - all_data["YearRemodAdd"]

all_data["TotalSF"] = (
    all_data["TotalBsmtSF"] +
    all_data["1stFlrSF"] +
    all_data["2ndFlrSF"]
)

qual_map = {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0}
qual_cols = [
    "ExterQual", "ExterCond", "BsmtQual",
    "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual"
]

for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map)


# ==============================
# Handle Skewness
# ==============================
skewed_feats = all_data.select_dtypes(include=np.number) \
                       .apply(lambda x: skew(x)) \
                       .sort_values(ascending=False)

skewed_feats = skewed_feats[skewed_feats > 0.75].index
print("\nNumber of skewed features:", len(skewed_feats))

all_data[skewed_feats] = np.log1p(all_data[skewed_feats])


# ==============================
# One-Hot Encoding
# ==============================
all_data = pd.get_dummies(all_data)

X_train = all_data.iloc[:len(y)]
X_test  = all_data.iloc[len(y):]

print("\nFinal train shape:", X_train.shape)
print("Final test shape:", X_test.shape)


# ==============================
# Models & Cross-Validation
# ==============================
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_rmse = np.sqrt(-cross_val_score(
    rf, X_train, y, cv=5,
    scoring="neg_mean_squared_error"
).mean())
print("Random Forest RMSE:", rf_rmse)


ridge = Ridge(alpha=15)
ridge_rmse = np.sqrt(-cross_val_score(
    ridge, X_train, y, cv=5,
    scoring="neg_mean_squared_error"
).mean())
print("Ridge RMSE:", ridge_rmse)


xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse = np.sqrt(-cross_val_score(
    xgb, X_train, y, cv=5,
    scoring="neg_mean_squared_error"
).mean())
print("XGBoost RMSE:", xgb_rmse)


lgbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_rmse = np.sqrt(-cross_val_score(
    lgbm, X_train, y, cv=5,
    scoring="neg_mean_squared_error"
).mean())
print("LightGBM RMSE:", lgbm_rmse)


# ==============================
# Model Comparison Plot
# ==============================
model_names = ["Ridge", "XGBoost", "LightGBM", "Random Forest"]
rmse_scores = [ridge_rmse, xgb_rmse, lgbm_rmse, rf_rmse]

plt.figure(figsize=(8, 5))
sns.barplot(x=model_names, y=rmse_scores)
plt.ylabel("RMSE (lower is better)")
plt.title("Model Comparison (CV RMSE)")
plt.show()


# ==============================
# Train Final Model & Predict
# ==============================
xgb.fit(X_train, y)
final_preds = np.expm1(xgb.predict(X_test))


# ==============================
# Submission
# ==============================
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_preds
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission saved as submission.csv")
